# Sesión 4 · Ver los datos

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución
→ **R**, y después Archivo → Guardar una copia en Drive.

In [ ]:
library(tidyverse)

In [ ]:
URL <- "https://docs.google.com/spreadsheets/d/e/2PACX-1vTTKC57eWZVIQ9lwhoR5nVqYM4kgi5zA9yifa-YStdfdJwNe7ATs0p-TUCUwvjfcWmHmvsZDEK8VX4I/pub?gid=1326008067&single=true&output=csv"

grupos <- read_csv(URL) |>
  select(
    momento  = Timestamp,
    genero   = `Género`,
    carrera  = Carrera,
    edad     = `Edad, en años cumplidos`,
    estatura = `Estatura, en metros`,
    traslado = `Tiempo de traslado a la universidad, en minutos`,
    calzado  = `Número de calzado`
  ) |>
  mutate(momento = mdy_hms(momento))

grupos

## 1 · Qué tipo de dato tengo

Antes de dibujar nada hay que saber con qué se está trabajando.

In [ ]:
glimpse(grupos)

Una variable es **categórica** si solo puede tomar uno de un conjunto pequeño de
valores. Es **numérica** si puede tomar un rango amplio y tiene sentido sumarla,
restarla o promediarla.

| Familia | Subtipo | Qué la distingue | En la encuesta |
|---|---|---|---|
| Categórica | Nominal | Sin orden | género, carrera |
| | Ordinal | Con orden, sin magnitud | nivel de estudios, NSE |
| Numérica | Discreta | Solo ciertos valores | edad, calzado |
| | Continua | Cualquier valor de un intervalo | estatura, traslado |

**Para discutir:** el número de calzado es un número y R puede promediarlo. ¿Qué
significaría el calzado promedio del salón?

El tipo de dato decide qué gráfica corresponde. Es una decisión tuya, no del
programa.

## 2 · Contar: la tabla de frecuencias

Antes de graficar una variable categórica, se cuenta.

In [ ]:
count(grupos, carrera)

In [ ]:
grupos |>
  count(carrera) |>
  mutate(proporcion = round(n / sum(n), 3))

## 3 · Una variable categórica

### Cada punto, una persona

Cada punto es un integrante del grupo. La altura de la pila es cuántos
contestaron lo mismo, es decir, la **frecuencia** de esa categoría.

In [ ]:
ggplot(grupos, aes(x = carrera)) +
  geom_dotplot(binaxis = "x", stackdir = "up", dotsize = 0.5) +
  scale_y_continuous(NULL, breaks = NULL) +
  labs(title = "Un punto por cada integrante del grupo", x = NULL)

### La misma información, como barra

**Para discutir:** ¿qué se gana al pasar de los puntos a la barra? ¿Qué se pierde?

In [ ]:
ggplot(grupos, aes(y = carrera)) +
  geom_bar() +
  labs(title = "Cómo está compuesto el grupo", x = "Estudiantes", y = NULL)

Se gana legibilidad: con trescientos respondientes la pila de puntos es
ilegible y la barra sigue funcionando.

Se pierde la unidad de observación. La barra es un rectángulo; el punto era una
persona que contestó.

La barra ya es un resumen: entre el dato y la barra hubo un conteo que descartó
todo lo demás que esa persona respondió.

### La gráfica de pastel

**Para discutir, antes de correr la celda de abajo:** mirando el pastel, ¿cuál
categoría es mayor y por cuánto?

In [ ]:
grupos |>
  count(carrera) |>
  ggplot(aes(x = "", y = n, fill = carrera)) +
  geom_col(width = 1) +
  coord_polar("y") +
  labs(x = NULL, y = NULL, fill = NULL) +
  theme_void()

Cleveland y McGill midieron con qué exactitud las personas leen cada codificación
visual. La **posición sobre una escala común**, que usa la barra, se juzga con
mucha mayor precisión que el **ángulo** y el **área**, que usa el pastel.

El pastel funciona con dos o tres categorías que sean fracciones simples. Con
más, obliga a comparar ángulos, y ahí el ojo falla. Este curso no las construye,
y sí las critica: vas a encontrarlas en informes todo el tiempo.

> Cleveland, W. S. y McGill, R. (1984). Graphical Perception. *JASA*, 79(387),
> 531-554.

## 4 · Una variable numérica

### Histograma

La barra de un histograma cubre un **intervalo** de valores, no una categoría.
Por eso las barras se tocan: el eje es continuo.

In [ ]:
ggplot(grupos, aes(x = estatura)) +
  geom_histogram(binwidth = 0.05, color = "white") +
  labs(title = "Estatura del grupo", x = "Estatura (m)", y = "Personas")

### El ancho del intervalo cambia la historia

Corre las dos celdas siguientes y compara. Los mismos datos, dos anchos, dos
formas distintas. Ninguna de las dos es falsa.

In [ ]:
ggplot(grupos, aes(x = estatura)) +
  geom_histogram(binwidth = 0.02, color = "white") +
  labs(x = "Estatura (m) · binwidth = 0.02", y = NULL)

In [ ]:
ggplot(grupos, aes(x = estatura)) +
  geom_histogram(binwidth = 0.10, color = "white") +
  labs(x = "Estatura (m) · binwidth = 0.10", y = NULL)

### Polígono de frecuencias

Une los puntos medios de las barras del histograma. Sirve para comparar dos
distribuciones en la misma gráfica, cosa que con barras encimadas no se puede.

In [ ]:
ggplot(grupos, aes(x = estatura)) +
  geom_freqpoly(binwidth = 0.05, linewidth = 1) +
  labs(x = "Estatura (m)", y = "Personas")

### Ojiva: la frecuencia acumulada

A cada estatura responde qué proporción del grupo mide eso o menos.

In [ ]:
ggplot(grupos, aes(x = estatura)) +
  stat_ecdf(linewidth = 1) +
  labs(x = "Estatura (m)", y = "Proporción acumulada")

## 5 · Dos variables

### Una categórica y una numérica: diagrama de caja

La caja va del primer al tercer cuartil y la línea de en medio es la mediana.

In [ ]:
ggplot(grupos, aes(x = estatura, y = genero)) +
  geom_boxplot() +
  labs(x = "Estatura (m)", y = NULL)

### Dos numéricas: gráfica de dispersión

**Para discutir:** si la nube sube de izquierda a derecha, ¿qué se puede afirmar?
¿Que ser alto causa tener el pie grande?

In [ ]:
ggplot(grupos, aes(x = estatura, y = calzado)) +
  geom_point(aes(color = genero), size = 3) +
  labs(x = "Estatura (m)", y = "Calzado", color = NULL)

## 6 · La gráfica de línea y las series de tiempo

Cuántas respuestas llegaron por minuto.

In [ ]:
grupos |>
  count(minuto = floor_date(momento, "minute")) |>
  ggplot(aes(x = minuto, y = n)) +
  geom_line(linewidth = 1) +
  geom_point(size = 2) +
  labs(x = NULL, y = "Respuestas")

Una **serie de tiempo** es una secuencia de observaciones de la misma variable,
ordenadas en el tiempo, donde cada valor depende de los anteriores y el conjunto
suele tener tendencia.

La gráfica de arriba cuenta respuestas por minuto de un evento que duró unos
minutos: tiene eje temporal, y no tiene ni tendencia ni dependencia entre
observaciones. Todavía no es una serie de tiempo.

El precio de una acción, el tipo de cambio y la inflación mensual sí lo son.
Sobre ellos el promedio del **nivel** dice poco, porque la serie sube y baja: se
trabaja con la variación de un periodo al siguiente.

Este curso trabaja con observaciones independientes entre sí. Las series de
tiempo quedan fuera, y también de *Análisis de Datos II*.

## Qué gráfica va con qué variable

| Tienes | Gráfica | En R |
|---|---|---|
| Una categórica | Barras, puntos | `geom_bar()`, `geom_dotplot()` |
| Una numérica | Histograma, polígono, ojiva | `geom_histogram()`, `geom_freqpoly()`, `stat_ecdf()` |
| Categórica y numérica | Diagrama de caja | `geom_boxplot()` |
| Dos numéricas | Dispersión | `geom_point()` |
| Una numérica en el tiempo | Línea | `geom_line()` |

Se lee de izquierda a derecha: primero se determina el tipo de variable, después
se elige la gráfica.

## Tarea

1. Del [catálogo de datos](https://cjjmdata.github.io/analisis_datos_i/datos/catalogo.html),
   elige una fuente de tu carrera y clasifica sus variables: cuáles son
   categóricas y cuáles numéricas, con su subtipo.
2. Busca una gráfica publicada en un informe o reporte de tu área y responde por
   escrito:
   - ¿Qué tipo de variable representa, y la gráfica le corresponde?
   - ¿Qué decisión tomó quien la hizo que pudo haber sido otra?
   - ¿Qué queda fuera que haría falta para juzgar lo que afirma?
3. Reproduce dos gráficas de hoy con los datos del grupo y escribe una línea
   sobre qué muestra cada una.

**Guarda tu copia.**